# Unlearning ViT-B/16 — fine-tuning complet (repli Kaggle)

Notebook autonome (session d'Orehn). **Réglages Kaggle : Accelerator = GPU (T4 ou P100), Internet = On.** Puis « Run All ».

Étapes : (1) télécharge `model.pt` et `forget_images.npz` depuis `SprintML/unlearning` ; (2) construit le professeur « jamais vu » à partir du ViT lui-même (sondes en validation croisée sur le CLS des blocs 8/9/10 + prédictions de l'original sous fortes perturbations) ; (3) télécharge des images IA publiques comme images de remplacement (retain) ; (4) fine-tuning complet : KL à T=2 vers les cibles sur des vues augmentées des 3 500 images, KL vers l'original sur les images de remplacement ; (5) sauvegarde un checkpoint par variante dans `/kaggle/working/` (float32, strict=True, < 450 Mo) + `results.json`.

Durée indicative sur T4 : préparation ~10 min, puis ~25 à 40 min par variante. **Ce notebook ne soumet rien.**

In [ ]:
import os
SMOKE = os.environ.get("SMOKE", "0") == "1"          # test rapide du code (quelques images, 2 pas)
WORK = os.environ.get("WORK", "/kaggle/working")      # sorties
DATA = os.environ.get("DATA", WORK)                   # model.pt / forget_images.npz (téléchargés si absents)
N_PROXY = 60 if SMOKE else 12000                      # images de remplacement (les 1000 dernières servent d'évaluation)
BS = 4 if SMOKE else 48                               # par flux (48 images à oublier + 48 de remplacement) ; mettre 32 si mémoire GPU insuffisante
# variantes : (nom, A, alpha, lr, pas).  A = probabilité moyenne (T=2) visée sur la vraie classe ; plus A est bas, plus on « oublie » fort.
# Ordre de soumission conseillé : du plus prudent au plus fort, s'arrêter quand le score officiel baisse.
VARIANTS = [("kg_A08_a25", 0.8, 2.5, 5e-5, 2000), ("kg_A07_a25", 0.7, 2.5, 5e-5, 2000), ("kg_A07_a4", 0.7, 4.0, 5e-5, 2000), ("kg_A06_a4", 0.6, 4.0, 5e-5, 2000)]
if SMOKE: VARIANTS = [("smoke", 0.7, 4.0, 5e-5, 2)]
GAMMA = 2.0        # poids de la KL vers l'original (retain)
L2SP = 0.0         # pénalité L2 vers les poids d'origine (0 = désactivée ; essayer 1e-3 à 1e-2 si le contrôle se dégrade)
os.makedirs(WORK, exist_ok=True)

In [ ]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"]); import timm
import math, json, time, copy, io, tarfile, shutil
import numpy as np, torch, torch.nn.functional as F
from huggingface_hub import hf_hub_download, hf_hub_url
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "timm", timm.__version__, "device", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")
assert dev == "cuda" or SMOKE, "Activer le GPU dans les réglages du notebook"
for f in ("model.pt", "forget_images.npz"):
    if not os.path.exists(f"{DATA}/{f}"):
        shutil.copy(hf_hub_download("SprintML/unlearning", f, repo_type="dataset"), f"{DATA}/{f}")

def build_model():
    return timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=100, init_values=1e-4)
def load_model(path=f"{DATA}/model.pt"):
    m = build_model(); m.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True); return m.to(dev).eval()
def preprocess(u8):   # identique au serveur
    return (u8.to(dev).permute(0, 3, 1, 2).float() / 255.0 - 0.5) / 0.5
def amp(): return torch.autocast(device_type=dev, dtype=torch.float16, enabled=dev == "cuda")

imgs = torch.from_numpy(np.load(f"{DATA}/forget_images.npz")["images"])
if SMOKE: imgs = imgs[:: len(imgs) // 60][:60]
N = len(imgs); orig = load_model()
for p in orig.parameters(): p.requires_grad_(False)
print("images à oublier :", tuple(imgs.shape))

In [ ]:
# ---- transformations ----
def affine(x, scale, tx, ty, rot, flip):
    B = len(x); cos, sin = torch.cos(rot) * scale, torch.sin(rot) * scale
    th = torch.zeros(B, 2, 3, device=x.device)
    th[:, 0, 0] = cos * flip; th[:, 0, 1] = -sin; th[:, 0, 2] = tx
    th[:, 1, 0] = sin * flip; th[:, 1, 1] = cos; th[:, 1, 2] = ty
    g = F.affine_grid(th, x.shape, align_corners=False)
    return F.grid_sample(x, g, mode="bilinear", padding_mode="reflection", align_corners=False)
def blur(x, s):
    return F.interpolate(F.interpolate(x, size=s, mode="bilinear", antialias=True), size=224, mode="bilinear")
def perturb(x, kind, g):
    # fortes perturbations : sortent l'image de la zone mémorisée, pour lire le comportement « non mémorisé » de l'original
    B, d = len(x), x.device
    U = lambda a, b: torch.rand(B, generator=g, device=d) * (b - a) + a
    one = torch.ones(B, device=d); zero = torch.zeros(B, device=d)
    if kind == "noise25": return x + 0.25 * torch.randn(x.shape, generator=g, device=d)
    if kind == "noise35": return x + 0.35 * torch.randn(x.shape, generator=g, device=d)
    if kind == "lowc_up": return x * 0.7 + 0.15
    if kind == "lowc_dn": return x * 0.7 - 0.15
    if kind == "highc": return (x * 1.35).clamp(-1, 1)
    if kind == "desat": return 0.3 * x + 0.7 * x.mean(1, keepdim=True)
    if kind == "rrc":
        s = U(0.55, 0.75); return affine(x, s, U(-1, 1) * (1 - s), U(-1, 1) * (1 - s), zero, one)
    if kind == "rot": return affine(x, one * 0.85, zero, zero, U(0.2, 0.4) * torch.sign(U(-1, 1)), one)
    if kind == "blur": return blur(x, 64)
    if kind == "cutout":
        m = (torch.rand(B, 1, 14, 14, generator=g, device=d) > 0.4).float()
        return x * F.interpolate(m, size=224, mode="nearest")
    raise ValueError(kind)
PERTS = ["noise25", "noise35", "lowc_up", "lowc_dn", "highc", "desat", "rrc", "rrc", "rot", "rot", "blur", "cutout", "cutout"]
def train_aug(x, g, strength=1.0):
    # vues modérées des images à oublier (tiennent lieu des copies altérées cachées) ; 25 % restent propres
    B, d = len(x), x.device
    U = lambda a, b: torch.rand(B, generator=g, device=d) * (b - a) + a
    clean = (U(0, 1) < 0.25).float()
    s = 1 - U(0, 0.3) * strength; flip = torch.where(U(0, 1) < 0.5, -1.0, 1.0)
    rot = U(-0.2, 0.2) * strength * (U(0, 1) < 0.3).float()
    y_ = affine(x, s, U(-1, 1) * (1 - s), U(-1, 1) * (1 - s), rot, flip)
    c = (1 + U(-0.25, 0.25) * strength).view(B, 1, 1, 1); b = (U(-0.15, 0.15) * strength).view(B, 1, 1, 1)
    y_ = y_ * c + b
    sat = (1 - U(0, 0.6) * (U(0, 1) < 0.3).float() * strength).view(B, 1, 1, 1)
    y_ = sat * y_ + (1 - sat) * y_.mean(1, keepdim=True)
    y_ = y_ + (U(0, 0.15) * (U(0, 1) < 0.5).float() * strength).view(B, 1, 1, 1) * torch.randn(x.shape, generator=g, device=d)
    bl = (U(0, 1) < 0.15 * strength).view(B, 1, 1, 1).float()
    y_ = (bl * blur(y_, 112) + (1 - bl) * y_).clamp(-1, 1)
    cl = clean.view(B, 1, 1, 1)
    return cl * x + (1 - cl) * y_

In [ ]:
# ---- professeur « jamais vu » ----
@torch.no_grad()
def orig_pass(kind=None, g=None, feats=False, bs=100):
    Z, Fs = [], []
    for i in range(0, N, bs):
        x = preprocess(imgs[i:i+bs])
        if kind: x = perturb(x, kind, g)
        with amp():
            h = orig.patch_embed(x); h = orig._pos_embed(h); h = orig.norm_pre(h); cl = []
            for b in orig.blocks:
                h = b(h); cl.append(h[:, 0].float())
            z = orig.forward_head(orig.norm(h))
        Z.append(z.float().cpu())
        if feats: Fs.append(torch.stack([cl[8], cl[9], cl[10]], 1).cpu())
    return torch.cat(Z), (torch.cat(Fs) if feats else None)
z0, cls_feats = orig_pass(feats=True); y = z0.argmax(1); idx = torch.arange(N)
print("classes présentes :", (torch.bincount(y, minlength=100) > 0).sum().item(), "| confiance moyenne de l'original :", z0.softmax(1).max(1).values.mean().item())
g = torch.Generator(device=dev).manual_seed(1); R = torch.zeros(N, 100); Py = []
for kind in PERTS:
    z, _ = orig_pass(kind, g); p2 = (z / 2).softmax(1); R += p2 / len(PERTS); Py.append(p2[idx, y])
    print(f"{kind:8s} accord={(z.argmax(1)==y).float().mean():.3f}")
Py = torch.stack(Py, 1)
perm = torch.randperm(N, generator=torch.Generator().manual_seed(0)); fold = torch.empty(N, dtype=torch.long); fold[perm] = torch.arange(N) % 5
def cvlog(X, wd):
    # régression logistique en validation croisée 5 plis : prédiction sur des images que la sonde n'a pas vues
    X = X.to(dev); X = (X - X.mean(0)) / (X.std(0) + 1e-6); P = torch.zeros(N, 100, device=dev); yc = y.to(dev); fd = fold.to(dev)
    for k in range(5):
        tr, te = fd != k, fd == k
        W = torch.zeros(X.shape[1], 100, device=dev, requires_grad=True); b = torch.zeros(100, device=dev, requires_grad=True)
        opt = torch.optim.LBFGS([W, b], lr=1, max_iter=20 if SMOKE else 200, history_size=20, line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad(); l = F.cross_entropy(X[tr] @ W + b, yc[tr]) + wd * W.pow(2).sum(); l.backward(); return l
        opt.step(closure)
        with torch.no_grad(): P[te] = (X[te] @ W + b).softmax(1)
    return P.cpu()
probes = {}
for j, l in enumerate((8, 9, 10)):
    best = None
    for wd in (1e-4, 1e-3, 1e-2):
        P = cvlog(cls_feats[:, j], wd); nll = -P[idx, y].clamp_min(1e-9).log().mean().item()
        if best is None or nll < best[0]: best = (nll, wd, P)
    probes[l] = best[2]; print(f"bloc {l} sonde CV : acc={(best[2].argmax(1)==y).float().mean():.4f} nll={best[0]:.4f}")
def lo(p): p = p.clamp(1e-4, 1 - 1e-4); return (p / (1 - p)).log()
zs = lambda v: (v - v.mean()) / v.std()
s_hard = zs(torch.stack([zs(lo(Py).mean(1)), zs(lo(probes[8][idx, y])), zs(lo(probes[9][idx, y])), zs(lo(probes[10][idx, y]))], 1).mean(1))   # grand = image facile
TEACH = {"y": y, "R": R, "P9": probes[9], "s": s_hard, "fold": fold}
torch.save(TEACH, f"{WORK}/teacher.pt")

def make_targets(T, A, alpha, wpert=0.5, wprobe=0.25):
    # cible à T=2 : q_i = pi_i * e_y + (1 - pi_i) * r_i ; pi_i = sigmoid(alpha * s_i + beta), beta tel que moyenne(pi) = A
    y = T["y"]
    def mask(P):
        P = P.clone(); P[idx, y] = 0; return P / P.sum(1, keepdim=True).clamp_min(1e-8)
    rp = mask(T["R"]); rb = mask(T["P9"])
    rc = torch.zeros(100, 100).index_add_(0, y, rp) / torch.bincount(y, minlength=100).clamp_min(1).unsqueeze(1)
    r = mask(wpert * rp + wprobe * rb + (1 - wpert - wprobe) * rc[y])
    lo_, hi_ = -30.0, 30.0
    for _ in range(60):
        beta = (lo_ + hi_) / 2
        if torch.sigmoid(alpha * T["s"] + beta).mean() < A: lo_ = beta
        else: hi_ = beta
    pi = torch.sigmoid(alpha * T["s"] + beta).clamp(1e-3, 0.995)
    q = (1 - pi).unsqueeze(1) * r; q[idx, y] = pi
    p1 = q.pow(2); p1 = p1 / p1.sum(1, keepdim=True); L = -p1[idx, y].log()
    print(f"cibles A={A} alpha={alpha} : acc visée={(q.argmax(1)==y).float().mean():.3f} perte visée moyenne={L.mean():.3f} écart-type={L.std():.3f}")
    return q

In [ ]:
# ---- images de remplacement (retain) : images IA publiques, lues en flux ----
from PIL import Image
import requests
ppath = f"{WORK}/proxies_{N_PROXY}.npz"
if os.path.exists(ppath):
    prox = torch.from_numpy(np.load(ppath)["images"])
else:
    url = hf_hub_url("jackyhate/text-to-image-2M", "data_512_2M/data_000000.tar", repo_type="dataset")
    r = requests.get(url, stream=True, timeout=60); r.raise_for_status()
    tf = tarfile.open(fileobj=r.raw, mode="r|"); ims = []
    for m in tf:
        if not m.name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")): continue
        try: im = Image.open(io.BytesIO(tf.extractfile(m).read())).convert("RGB")
        except Exception: continue
        w, h = im.size; s = min(w, h); im = im.crop(((w-s)//2, (h-s)//2, (w-s)//2+s, (h-s)//2+s)).resize((224, 224), Image.BICUBIC)
        ims.append(np.asarray(im))
        if len(ims) % 2000 == 0: print(len(ims), "images")
        if len(ims) >= N_PROXY: break
    r.close(); prox = torch.from_numpy(np.stack(ims)); np.savez(ppath, images=prox.numpy()); del ims
@torch.no_grad()
def logits(m, images, sel, fn=None, bs=100):
    out = []
    for i in range(0, len(sel), bs):
        x = preprocess(images[sel[i:i+bs]])
        if fn is not None: x = fn(x)
        with amp(): out.append(m(x).float())
    return torch.cat(out)
zprox = logits(orig, prox, torch.arange(len(prox))).cpu()
n_te = 20 if SMOKE else 1000
PTR, PTE = torch.arange(0, len(prox) - n_te), torch.arange(len(prox) - n_te, len(prox))
print("images de remplacement :", len(prox), "| classes touchées :", (torch.bincount(zprox.argmax(1), minlength=100) > 0).sum().item(), "| confiance moyenne :", zprox.softmax(1).max(1).values.mean().item())

In [ ]:
# ---- fine-tuning complet ----
def kl2(t, z, is_logits=False):
    # KL(cible || softmax(z/2)) par image ; cible = probabilités, ou logits de l'original si is_logits
    ls = (z.float() / 2).log_softmax(1); tq = (t.float() / 2).softmax(1) if is_logits else t
    return (tq * (tq.clamp_min(1e-9).log() - ls)).sum(1)
EVAL_VIEWS = {"propre": None, "miroir": lambda x: x.flip(-1), "contraste": lambda x: x * 0.8 + 0.1,
              "bruit": lambda x: x + 0.2 * torch.randn(x.shape, device=x.device, generator=torch.Generator(device=x.device).manual_seed(5)), "flou": lambda x: blur(x, 160)}
def finetune(name, A, alpha, lr, steps, bs=BS, gamma=GAMMA, l2sp=L2SP, seed=0):
    torch.manual_seed(seed); q = make_targets(TEACH, A, alpha); qd, yd = q.to(dev), y.to(dev)
    model = copy.deepcopy(orig).train()                      # ni dropout ni drop-path dans cette architecture
    params = list(model.parameters())
    for p in params: p.requires_grad_(True)
    p0 = [p.detach().clone() for p in params]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / 40) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))
    scaler = torch.amp.GradScaler(enabled=dev == "cuda"); g = torch.Generator(device=dev).manual_seed(seed)
    sub = torch.arange(0, N, max(1, N // 1000))
    @torch.no_grad()
    def evaluate(step):
        model.eval(); res = {"step": step}
        for vn, fn in EVAL_VIEWS.items():
            z = logits(model, imgs, sub, fn); L = F.cross_entropy(z, yd[sub], reduction="none")
            res[vn] = dict(acc=round((z.argmax(1) == yd[sub]).float().mean().item(), 4), Lmean=round(L.mean().item(), 3), Lstd=round(L.std().item(), 3), KL_cible=round(kl2(qd[sub], z).mean().item(), 4))
        z = logits(model, prox, PTE); zo = zprox[PTE].to(dev); k = kl2(zo, z, True); ag = (z.argmax(1) == zo.argmax(1)).float(); hc = zo.softmax(1).max(1).values > 0.9
        res["controle"] = dict(KL=round(k.mean().item(), 4), accord=round(ag.mean().item(), 4), accord_confiant=round(ag[hc].mean().item(), 4) if hc.any() else None)
        res["rel_dw"] = round((sum((p - p_).pow(2).sum() for p, p_ in zip(params, p0)).sqrt() / sum(p_.pow(2).sum() for p_ in p0).sqrt()).item(), 5)
        print(json.dumps(res, ensure_ascii=False), flush=True); model.train(); return res
    res = evaluate(0); t0 = time.time()
    for step in range(1, steps + 1):
        bi = torch.randint(N, (bs,)); pi = PTR[torch.randint(len(PTR), (bs,))]
        xf = train_aug(preprocess(imgs[bi]), g); xp = preprocess(prox[pi])
        fl = torch.rand(bs, device=dev, generator=g) < 0.5; xp = torch.where(fl.view(-1, 1, 1, 1), xp.flip(-1), xp)
        with amp():
            with torch.no_grad(): zo = orig(xp)
            z = model(torch.cat([xf, xp]))
        lf = kl2(qd[bi], z[:bs]).mean(); lr_ = kl2(zo, z[bs:], True).mean()
        reg = sum((p - p_).pow(2).sum() for p, p_ in zip(params, p0)) if l2sp > 0 else torch.zeros((), device=dev)
        loss = lf + gamma * lr_ + l2sp * reg
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update(); sched.step()
        if step % 100 == 0: print(f"pas {step}/{steps} oubli={lf.item():.4f} retain={lr_.item():.4f} {time.time()-t0:.0f}s", flush=True)
        if step % 500 == 0 or step == steps: res = evaluate(step)
    sd = {k: v.detach().cpu().float() for k, v in model.state_dict().items()}
    assert all(torch.isfinite(v).all() for v in sd.values())
    out = f"{WORK}/{name}.pt"; torch.save(sd, out)
    chk = build_model(); chk.load_state_dict(torch.load(out, map_location="cpu", weights_only=True), strict=True)
    assert all(v.dtype == torch.float32 for v in chk.state_dict().values() if v.is_floating_point()) and os.path.getsize(out) < 450e6
    print("sauvegardé :", out, round(os.path.getsize(out) / 1e6, 1), "Mo", flush=True)
    del model, opt, p0; torch.cuda.empty_cache()
    return res

RESULTS = {}
for (name, A, alpha, lr, steps) in VARIANTS:
    print("=" * 30, name); RESULTS[name] = dict(A=A, alpha=alpha, lr=lr, steps=steps, final=finetune(name, A, alpha, lr, steps))
    json.dump(RESULTS, open(f"{WORK}/results.json", "w"), indent=1, ensure_ascii=False)

In [ ]:
# ---- résumé ----
print(f"{'variante':14s} {'acc':>6s} {'perte moy/é.-t.':>16s} {'KL cible':>9s} {'contrôle KL':>12s} {'accord':>7s} {'écart poids':>11s}")
for n_, r in RESULTS.items():
    f, c = r["final"]["propre"], r["final"]["controle"]
    print(f"{n_:14s} {f['acc']:6.3f} {f['Lmean']:8.3f}/{f['Lstd']:<7.3f} {f['KL_cible']:9.4f} {c['KL']:12.4f} {c['accord']:7.3f} {r['final']['rel_dw']:11.4f}")
print("Fichiers à récupérer dans", WORK, ":", [f for f in os.listdir(WORK) if f.endswith(".pt") and f != "teacher.pt"])